In [1]:
import pandas as pd
import requests
import time
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
import yfinance as yf

# ── Config ─────────────────────────────────────────────
load_dotenv()
DATA_DIR     = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "NVDA", "AVGO", "MU",   "AMD",  "AMAT"
]

TICKER_GROUPS = {
    "AAPL": "tech", "MSFT": "tech", "GOOGL": "tech",
    "AMZN": "tech", "META": "tech",
    "NVDA": "semi", "AVGO": "semi", "MU":   "semi",
    "AMD":  "semi", "AMAT": "semi"
}

In [2]:
def fetch_alpha_vantage_earnings(tickers, api_key):
    """
    Queries Alpha Vantage for historical actual vs estimated EPS data.
    """
    all_records = []
    
    for ticker in tickers:
        print(f"Fetching historical earnings from Alpha Vantage for: {ticker}...")
        
        # Alpha Vantage Fundamental Earnings Endpoint
        url = f"https://www.alphavantage.co/query?function=EARNINGS&symbol={ticker}&apikey={api_key}"
        
        try:
            response = requests.get(url)
            
            if response.status_code != 200:
                print(f"  -> Error HTTP {response.status_code}")
                continue
                
            data = response.json()
            
            # Check for API rate limit message or bad key
            if "Note" in data:
                print("  -> Warning: Hitting Alpha Vantage frequency limits. Slowing down...")
                time.sleep(30)
                continue
                
            if "quarterlyEarnings" in data:
                df = pd.DataFrame(data["quarterlyEarnings"])
                df['ticker'] = ticker
                all_records.append(df)
                print(f"  -> Successfully retrieved {len(df)} quarters of data.")
            else:
                print(f"  -> No data found for {ticker}. Response keys: {list(data.keys())}")
                
        except Exception as e:
            print(f"  -> Exception occurred: {e}")
            
        # CRITICAL: Alpha Vantage free tier allows 5 requests/min. 
        # 15 seconds ensures we stay under that limit safely.
        print("  -> Waiting 15 seconds for API rate limit limits...")
        time.sleep(15)
        
    if not all_records:
        return pd.DataFrame()
        
    # Combine everything together
    full_df = pd.concat(all_records, ignore_index=True)
    
    # Keep and clean relevant columns
    # AV returns: fiscalDateEnding, reportedDate, reportedEPS, estimatedEPS, surprise, surprisePercentage
    columns_map = {
        'ticker': 'ticker',
        'reportedDate': 'earnings_date',
        'reportedEPS': 'actual_eps',
        'estimatedEPS': 'consensus_eps',
        'fiscalDateEnding': 'fiscal_date_ending'
    }
    
    full_df = full_df.rename(columns=columns_map)
    available_cols = [c for c in columns_map.values() if c in full_df.columns]
    full_df = full_df[available_cols]
    
    # Clean and force numeric formatting
    full_df['earnings_date'] = pd.to_datetime(full_df['earnings_date']).dt.date
    full_df['actual_eps'] = pd.to_numeric(full_df['actual_eps'], errors='coerce')
    full_df['consensus_eps'] = pd.to_numeric(full_df['consensus_eps'], errors='coerce')
    
    # Drop rows missing crucial modeling variables
    full_df = full_df.dropna(subset=['actual_eps', 'consensus_eps'])
    
    # Filter for the last 5 years (2021 onward)
    five_years_ago = datetime.today().date() - pd.Timedelta(days=5*365)
    full_df = full_df[full_df['earnings_date'] >= five_years_ago]
    
    # Group tracking alignment
    full_df['sector_group'] = full_df['ticker'].map(TICKER_GROUPS)
    
    # Calculate target labels (1 = Beat/Met, 0 = Miss)
    full_df['surprise_amount'] = full_df['actual_eps'] - full_df['consensus_eps']
    full_df['target_label'] = full_df['surprise_amount'].apply(lambda x: 1 if x >= 0 else 0)
    
    return full_df.sort_values(by=['ticker', 'earnings_date'], ascending=[True, False]).reset_index(drop=True)

if __name__ == "__main__":
  
    earnings_df = fetch_alpha_vantage_earnings(TICKERS, load_dotenv())
        
    if not earnings_df.empty:
        print(f"\nSuccess! Collected {len(earnings_df)} clean historical data rows.")
        print(earnings_df.head(10))
            
        output_path = os.path.join(DATA_DIR, "earnings_base.csv")
        earnings_df.to_csv(output_path, index=False)
        print(f"\nSaved master target dataframe to: '{output_path}'")
    else:
        print("\nCould not retrieve historical data. Ensure your API key is active.")

Fetching historical earnings from Alpha Vantage for: AAPL...
  -> Successfully retrieved 121 quarters of data.
  -> Waiting 15 seconds for API rate limit limits...
Fetching historical earnings from Alpha Vantage for: MSFT...
  -> Successfully retrieved 121 quarters of data.
  -> Waiting 15 seconds for API rate limit limits...
Fetching historical earnings from Alpha Vantage for: GOOGL...
  -> Successfully retrieved 87 quarters of data.
  -> Waiting 15 seconds for API rate limit limits...
Fetching historical earnings from Alpha Vantage for: AMZN...
  -> Successfully retrieved 116 quarters of data.
  -> Waiting 15 seconds for API rate limit limits...
Fetching historical earnings from Alpha Vantage for: META...
  -> Successfully retrieved 57 quarters of data.
  -> Waiting 15 seconds for API rate limit limits...
Fetching historical earnings from Alpha Vantage for: NVDA...
  -> Successfully retrieved 109 quarters of data.
  -> Waiting 15 seconds for API rate limit limits...
Fetching historic

In [3]:
# ── Config ─────────────────────────────────────────────
DATA_DIR = "../data/raw"
BASE_CSV = os.path.join(DATA_DIR, "earnings_base.csv")
PRICE_DIR = os.path.join(DATA_DIR, "ticker_prices")
os.makedirs(PRICE_DIR, exist_ok=True)

WINDOW_BEFORE = 30  # Number of days to pull before the earnings announcement

def download_market_features(base_csv_path):
    """
    Reads the base earnings file and downloads historical stock prices
    spanning the required lookback windows for each ticker.
    """
    if not os.path.exists(base_csv_path):
        print(f"Error: Base file not found at {base_csv_path}. Please run your Alpha Vantage script first.")
        return
        
    # 1. Load your earnings data
    df_earnings = pd.read_csv(base_csv_path)
    df_earnings['earnings_date'] = pd.to_datetime(df_earnings['earnings_date'])
    
    # Extract the unique list of tickers we need data for
    unique_tickers = df_earnings['ticker'].unique()
    print(f"Found {len(unique_tickers)} tickers in your base CSV file.")
    
    # 2. Find the absolute minimum and maximum date ranges needed across all stocks
    # This prevents hitting yfinance thousands of times in a slow loop
    global_start = df_earnings['earnings_date'].min() - timedelta(days=WINDOW_BEFORE + 5)
    global_end = df_earnings['earnings_date'].max() + timedelta(days=5)
    
    print(f"Downloading historical blocks from {global_start.date()} to {global_end.date()}...")
    
    # 3. Download data ticker by ticker
    for ticker in unique_tickers:
        print(f"Downloading historical price framework for: {ticker}...")
        
        try:
            # Fetch data from Yahoo Finance
            # yfinance automatically handles adjustments for stock splits/dividends out of the box
            stock_data = yf.download(ticker, start=global_start, end=global_end, progress=False)
            
            if stock_data.empty:
                print(f"  -> Warning: No market pricing returned for {ticker}")
                continue
                
            # Reset index to turn the Date from an index into a standard column
            stock_data = stock_data.reset_index()
            
            # Clean up the column naming convention (sometimes yfinance yields MultiIndex columns)
            if isinstance(stock_data.columns, pd.MultiIndex):
                stock_data.columns = stock_data.columns.droplevel(1)
                
            stock_data.columns = [col.lower() for col in stock_data.columns]
            
            # Save the raw time series block to its own file
            ticker_file = os.path.join(PRICE_DIR, f"{ticker}_daily_prices.csv")
            stock_data.to_csv(ticker_file, index=False)
            print(f"  -> Saved {len(stock_data)} days of price data to '{ticker_file}'")
            
        except Exception as e:
            print(f"  -> Failed to pull market metrics for {ticker}: {e}")

if __name__ == "__main__":
    download_market_features(BASE_CSV)
    print("\nMarket data gathering phase complete!")

Found 10 tickers in your base CSV file.
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/AAPL_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/AMAT_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/AMD_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/AMZN_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/AVGO_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/GOOGL_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/META_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/MSFT_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/MU_daily_prices.csv'
  -> Saved 1263 days of price data to '../data/raw/ticker_prices/NVDA_daily_prices.csv'

Market data gathering phase complete!
